# Phase 1: Data Audit + Normalization
### Multi-Source Entity Resolution (Source 1, Source 2, Source 3)

**Goal:** Turn messy, inconsistent raw text into clean, comparable fields — without throwing away the original.

**Scope:**
- Applies to all 6 files: `train_source1`, `train_source2`, `train_source3`, `test_source1`, `test_source2`, `test_source3`.
- **Step 1 — Audit First:**
  1. Pull 100–200 known-matching pairs from `train_ground_truth.tsv` and inspect variations side-by-side (S1 vs matched S2/S3).
  2. Audit name patterns, address noise, missing fields, legal suffixes, and country distributions across S1, S2, and S3.
  3. Compile findings into a baseline audit report.
- **Step 2 — Normalize Using Findings:**
  1. `unicode_utils`: Mojibake cleanup, NFKD normalization, accent stripping, control character removal.
  2. `name_normalizer`: Strips/standardizes legal suffixes (Ltd/Pvt/Inc/LLC), expands abbreviations, handles `&` vs `and`, punctuation.
  3. `address_normalizer`: Standardizes street abbreviations (St/Rd/Ave), handles landmark references ("Near SBI ATM"), expands unit designators.
  4. `country_normalizer`: Open-set canonical mapping (US, India, France, etc.) without hardcoding.
  5. `normalization_pipeline`: Chains unicode → name → address → country.
- **Critical Constraint:** Raw fields are **never overwritten**. Every record maintains `business_name_raw`, `business_address_raw`, and `country_raw` alongside their normalized counterparts.
- **Separate Handling:** Dedicated, modular normalization workflows for Source 1, Source 2, and Source 3.


In [ ]:
import os
import sys
import re
import unicodedata
import csv
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any

# Set project roots
ROOT_DIR = Path("..").resolve() if Path("..").resolve().name == "AMAZON_ML_CHALLENGE" else Path(".").resolve()
DATA_DIR = ROOT_DIR / "student_resource" / "dataset"
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

PROCESSED_DIR = ROOT_DIR / "data" / "processed" / "phase1"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

REPORTS_DIR = ROOT_DIR / "reports" / "phase1"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {ROOT_DIR}")
print(f"Train Dataset Dir: {TRAIN_DIR} (Exists: {TRAIN_DIR.exists()})")
print(f"Test Dataset Dir: {TEST_DIR} (Exists: {TEST_DIR.exists()})")
print(f"Output Processed Dir: {PROCESSED_DIR}")


## Step 1: Ground Truth Audit & Pair Sampling
Before writing cleanup rules, we inspect real ground-truth matching pairs from `train_ground_truth.tsv` side-by-side to observe:
- How legal suffixes vary ("Corp" vs "Corporation", "Pvt Ltd" vs "Private Limited")
- How street addresses vary ("Rd" vs "Road", "St" vs "Street", reordered numbers/units)
- Encoding issues & mojibake in Source 2 and Source 3
- Missing fields and abbreviation patterns


In [ ]:
def load_tsv_sample(file_path: Path, max_rows: int = 50000) -> Dict[str, Dict[str, str]]:
    """Loads a TSV file into an entity_id indexed dictionary for fast lookup."""
    records = {}
    with open(file_path, mode="r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for i, row in enumerate(reader):
            records[row["entity_id"]] = row
            if i + 1 >= max_rows:
                break
    return records

print("Loading sample partitions of S1, S2, S3 for Ground Truth Inspection...")
sample_s1 = load_tsv_sample(TRAIN_DIR / "train_source1.tsv", max_rows=50000)
sample_s2 = load_tsv_sample(TRAIN_DIR / "train_source2.tsv", max_rows=50000)
sample_s3 = load_tsv_sample(TRAIN_DIR / "train_source3.tsv", max_rows=50000)

print(f"Loaded: S1 sample={len(sample_s1):,}, S2 sample={len(sample_s2):,}, S3 sample={len(sample_s3):,}")


In [ ]:
def sample_ground_truth_pairs(gt_path: Path, n_samples: int = 150) -> List[Dict[str, Any]]:
    """Extracts ground truth matching pairs from train_ground_truth.tsv."""
    pairs = []
    with open(gt_path, mode="r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            s1_id = row["source1_entity_id"]
            matched_ids = [m.strip() for m in row["matched_entity_ids"].split(",") if m.strip()]
            
            if s1_id in sample_s1:
                s1_rec = sample_s1[s1_id]
                for match_id in matched_ids:
                    match_rec = None
                    source_label = "S2" if match_id.startswith("S2") else "S3"
                    if source_label == "S2" and match_id in sample_s2:
                        match_rec = sample_s2[match_id]
                    elif source_label == "S3" and match_id in sample_s3:
                        match_rec = sample_s3[match_id]
                        
                    if match_rec:
                        pairs.append({
                            "s1_id": s1_id,
                            "s1_name": s1_rec.get("business_name", ""),
                            "s1_addr": s1_rec.get("business_address", ""),
                            "s1_country": s1_rec.get("country", ""),
                            "match_source": source_label,
                            "match_id": match_id,
                            "match_name": match_rec.get("business_name", ""),
                            "match_addr": match_rec.get("business_address", ""),
                            "match_country": match_rec.get("country", "")
                        })
            if len(pairs) >= n_samples:
                break
    return pairs

gt_pairs = sample_ground_truth_pairs(TRAIN_DIR / "train_ground_truth.tsv", n_samples=20)
print(f"Found {len(gt_pairs)} matched pairs available in samples. Displaying first 10 side-by-side:")

print("="*100)
for idx, p in enumerate(gt_pairs[:10], 1):
    print(f"PAIR #{idx} [Source 1 ({p['s1_id']}) <--> {p['match_source']} ({p['match_id']})]")
    print(f"  NAME:      S1: '{p['s1_name']}'")
    print(f"             {p['match_source']}: '{p['match_name']}'")
    print(f"  ADDRESS:   S1: '{p['s1_addr']}'")
    print(f"             {p['match_source']}: '{p['match_addr']}'")
    print(f"  COUNTRY:   S1: '{p['s1_country']}' | {p['match_source']}: '{p['match_country']}'")
    print("-" * 100)


## Step 2: Full Field Audits across S1, S2, and S3
We audit the data for:
1. **Name Patterns:** Legal suffix frequencies, punctuation, casing, encoding corruption.
2. **Address Patterns:** Street abbreviations, landmark tokens, missing values.
3. **Country Patterns:** Categorical distribution and open-set values (ensuring non-US/India countries like France in test are handled).


In [ ]:
def run_field_audit(data_dict: Dict[str, Dict[str, str]], source_name: str, sample_size: int = 50000) -> Dict[str, Any]:
    """Audits name, address, and country fields for a given source."""
    total = len(data_dict)
    name_lengths = []
    missing_addr = 0
    missing_name = 0
    missing_country = 0
    
    # Common suffix patterns to check
    legal_patterns = ["inc", "ltd", "corp", "corporation", "llc", "pvt", "limited", "co", "company", "gmbh"]
    legal_counts = Counter()
    special_char_counts = Counter()
    countries = Counter()
    mojibake_count = 0
    
    for row in list(data_dict.values())[:sample_size]:
        name = row.get("business_name", "") or ""
        addr = row.get("business_address", "") or ""
        country = row.get("country", "") or ""
        
        if not name.strip():
            missing_name += 1
        if not addr.strip():
            missing_addr += 1
        if not country.strip():
            missing_country += 1
            
        name_lengths.append(len(name))
        
        # Check mojibake / corrupted encoding
        if "\ufffd" in name or "\ufffd" in addr or any(ord(c) == 65533 for c in name + addr):
            mojibake_count += 1
            
        # Legal suffixes in name
        name_lower = name.lower()
        for pat in legal_patterns:
            if re.search(r"\b" + pat + r"\b", name_lower):
                legal_counts[pat] += 1
                
        # Special characters
        for char in "&+/@#.,;:":
            if char in name:
                special_char_counts[char] += 1
                
        countries[country.strip()] += 1

    stats = {
        "source": source_name,
        "sample_size": min(total, sample_size),
        "missing_name": missing_name,
        "missing_addr": missing_addr,
        "missing_country": missing_country,
        "mojibake_count": mojibake_count,
        "top_legal_suffixes": legal_counts.most_common(5),
        "special_chars": special_char_counts.most_common(5),
        "top_countries": countries.most_common(5)
    }

    print(f"\n=== AUDIT REPORT FOR {source_name} (Sample: {stats['sample_size']:,} records) ===")
    print(f"- Missing Names:       {missing_name} ({missing_name/total*100:.2f}%)")
    print(f"- Missing Addresses:   {missing_addr} ({missing_addr/total*100:.2f}%)")
    print(f"- Missing Countries:   {missing_country} ({missing_country/total*100:.2f}%)")
    print(f"- Mojibake / Corrupted:{mojibake_count} records")
    print(f"- Top Legal Suffixes:  {stats['top_legal_suffixes']}")
    print(f"- Special Characters:  {stats['special_chars']}")
    print(f"- Country Breakdown:   {stats['top_countries']}")
    return stats

audit_s1 = run_field_audit(sample_s1, "Source 1")
audit_s2 = run_field_audit(sample_s2, "Source 2")
audit_s3 = run_field_audit(sample_s3, "Source 3")

# Write comprehensive audit report to reports/phase1/audit_report.md
audit_md_path = REPORTS_DIR / "audit_report.md"
with open(audit_md_path, "w", encoding="utf-8") as f_rep:
    f_rep.write("# Phase 1: Data Audit Report\n\n")
    f_rep.write("This report serves as the rulebook for Phase 1 normalization requirements.\n\n")
    for s in [audit_s1, audit_s2, audit_s3]:
        f_rep.write(f"## {s['source']} Audit Findings (Sample: {s['sample_size']:,} rows)\n")
        f_rep.write(f"- **Missing Names**: {s['missing_name']}\n")
        f_rep.write(f"- **Missing Addresses**: {s['missing_addr']}\n")
        f_rep.write(f"- **Missing Countries**: {s['missing_country']}\n")
        f_rep.write(f"- **Mojibake Records**: {s['mojibake_count']}\n")
        f_rep.write(f"- **Top Legal Suffixes**: `{s['top_legal_suffixes']}`\n")
        f_rep.write(f"- **Special Characters**: `{s['special_chars']}`\n")
        f_rep.write(f"- **Top Countries**: `{s['top_countries']}`\n\n")
print(f"\nSaved compiled audit findings to: {audit_md_path}")


## Step 3: Normalization Rules & Utilities
We construct modular, regex-compiled normalization components:
1. `unicode_utils`: Mojibake cleaning, NFKD decomposition, ASCII transliteration, whitespace normalization.
2. `name_normalizer`: Standardizes legal suffixes, expands business abbreviations, standardizes symbols (`&` -> `and`), cleans punctuation.
3. `address_normalizer`: Standardizes street abbreviations (`st` -> `street`, `rd` -> `road`, `ave` -> `avenue`), handles unit designators and landmarks.
4. `country_normalizer`: Open-set mapping that canonicalizes variants (`US`, `USA`, `India`, `IND`, `France`, etc.) while gracefully preserving unmapped countries.


In [ ]:
# ==============================================================================
# REFERENCE DICTIONARIES & COMPILED PATTERNS
# ==============================================================================

# Legal entity suffixes to standard canonical form
LEGAL_SUFFIX_MAP = {
    "private limited": "pvt ltd",
    "pvt. ltd.": "pvt ltd",
    "pvt ltd": "pvt ltd",
    "pvt.": "pvt",
    "pvt": "pvt",
    "limited": "ltd",
    "ltd.": "ltd",
    "ltd": "ltd",
    "incorporated": "inc",
    "inc.": "inc",
    "inc": "inc",
    "corporation": "corp",
    "corp.": "corp",
    "corp": "corp",
    "company": "co",
    "co.": "co",
    "co": "co",
    "limited liability company": "llc",
    "llc.": "llc",
    "llc": "llc",
    "l.l.c.": "llc",
    "l.l.c": "llc",
    "gesellschaft mit beschrankter haftung": "gmbh",
    "gmbh": "gmbh",
    "public limited company": "plc",
    "plc": "plc",
    "societe anonyme": "sa",
    "s.a.": "sa"
}

# Name abbreviation expansions
NAME_ABBREVIATIONS = {
    "intl": "international",
    "int'l": "international",
    "mfg": "manufacturing",
    "tech": "technology",
    "technologies": "technology",
    "svcs": "services",
    "svc": "service",
    "assoc": "associates",
    "grp": "group",
    "sys": "systems",
    "mgmt": "management",
    "hldg": "holding",
    "hldgs": "holdings",
    "distr": "distribution",
    "ent": "enterprise",
    "enterprises": "enterprise",
    "pharm": "pharmacy",
    "med": "medical",
    "ctr": "center",
    "lab": "laboratory",
    "labs": "laboratory"
}

# Address street suffix normalization
STREET_SUFFIX_MAP = {
    "st": "street", "st.": "street", "street": "street",
    "rd": "road", "rd.": "road", "road": "road",
    "ave": "avenue", "ave.": "avenue", "av": "avenue", "avenue": "avenue",
    "blvd": "boulevard", "blvd.": "boulevard", "boulevard": "boulevard",
    "dr": "drive", "dr.": "drive", "drive": "drive",
    "ln": "lane", "ln.": "lane", "lane": "lane",
    "ct": "court", "ct.": "court", "court": "court",
    "hwy": "highway", "hwy.": "highway", "highway": "highway",
    "pkwy": "parkway", "pkwy.": "parkway", "parkway": "parkway",
    "pl": "place", "pl.": "place", "place": "place",
    "sq": "square", "sq.": "square", "square": "square",
    "cir": "circle", "cir.": "circle", "circle": "circle",
    "ste": "suite", "ste.": "suite", "suite": "suite",
    "apt": "apartment", "apt.": "apartment", "apartment": "apartment",
    "fl": "floor", "fl.": "floor", "floor": "floor",
    "bldg": "building", "bldg.": "building", "building": "building",
    "kh no": "khasra number", "kh no.": "khasra number",
    "no": "number", "no.": "number"
}

# Directional abbreviations
DIRECTION_MAP = {
    "n": "north", "n.": "north",
    "s": "south", "s.": "south",
    "e": "east", "e.": "east",
    "w": "west", "w.": "west",
    "ne": "northeast", "ne.": "northeast",
    "nw": "northwest", "nw.": "northwest",
    "se": "southeast", "se.": "southeast",
    "sw": "southwest", "sw.": "southwest"
}

# Open-Set Country Mapping: Canonical mappings for known variants, open pass-through for unseen (e.g. France)
COUNTRY_MAP = {
    "us": "US", "usa": "US", "united states": "US", "united states of america": "US", "u.s.a.": "US", "u.s.": "US",
    "in": "India", "ind": "India", "india": "India", "republic of india": "India",
    "fr": "France", "fra": "France", "france": "France",
    "uk": "United Kingdom", "gbr": "United Kingdom", "united kingdom": "United Kingdom", "great britain": "United Kingdom",
    "ca": "Canada", "can": "Canada", "canada": "Canada",
    "de": "Germany", "deu": "Germany", "germany": "Germany"
}


In [ ]:
class UnicodeUtils:
    """Handles unicode cleanup, mojibake resolution, accent folding, and whitespace normalization."""
    
    RE_MOJIBAKE = re.compile(r"[\ufffd\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
    RE_WHITESPACE = re.compile(r"\s+")
    
    @classmethod
    def clean(cls, text: Optional[str]) -> str:
        if not text:
            return ""
        # 1. Strip replacement characters and non-printable control chars
        cleaned = cls.RE_MOJIBAKE.sub(" ", text)
        # 2. Decompose accented characters (NFKD) and strip non-spacing marks
        normalized = unicodedata.normalize("NFKD", cleaned)
        ascii_text = "".join(c for c in normalized if not unicodedata.combining(c))
        # 3. Collapse multiple whitespaces
        return cls.RE_WHITESPACE.sub(" ", ascii_text).strip()

class NameNormalizer:
    """Cleans, expands, and normalizes business names."""
    
    RE_AMP = re.compile(r"\s*&\s*")
    RE_PLUS = re.compile(r"\s*\+\s*")
    RE_AT = re.compile(r"\s*@\s*")
    RE_NOISE = re.compile(r"[^a-zA-Z0-9\s]")
    RE_SPACES = re.compile(r"\s+")
    
    @classmethod
    def normalize(cls, name: Optional[str]) -> str:
        if not name:
            return ""
        text = UnicodeUtils.clean(name).lower()
        
        # Symbol standardization
        text = cls.RE_AMP.sub(" and ", text)
        text = cls.RE_PLUS.sub(" and ", text)
        text = cls.RE_AT.sub(" at ", text)
        
        # Remove unwanted punctuation
        text = cls.RE_NOISE.sub(" ", text)
        tokens = text.split()
        
        # Standardize legal suffixes and abbreviations
        cleaned_tokens = []
        for token in tokens:
            if token in NAME_ABBREVIATIONS:
                token = NAME_ABBREVIATIONS[token]
            cleaned_tokens.append(token)
            
        full_text = " ".join(cleaned_tokens)
        
        # Match legal suffix at end of company name
        for suffix_raw, suffix_canonical in LEGAL_SUFFIX_MAP.items():
            pattern = r"\b" + re.escape(suffix_raw) + r"$"
            if re.search(pattern, full_text):
                full_text = re.sub(pattern, suffix_canonical, full_text)
                break
                
        return cls.RE_SPACES.sub(" ", full_text).strip()

class AddressNormalizer:
    """Cleans and standardizes addresses without discarding key positional indicators."""
    
    RE_LANDMARK = re.compile(r"\b(near|opp|opposite|behind|beside|adjacent to)\b\s*.*?(?=,|$)", re.IGNORECASE)
    RE_NOISE = re.compile(r"[^a-zA-Z0-9\s,]")
    RE_SPACES = re.compile(r"\s+")
    
    @classmethod
    def normalize(cls, address: Optional[str]) -> str:
        if not address:
            return ""
        text = UnicodeUtils.clean(address).lower()
        
        # Replace punctuation except commas with space
        text = cls.RE_NOISE.sub(" ", text)
        
        # Split by comma or whitespace for token-level expansion
        tokens = text.replace(",", " , ").split()
        normalized_tokens = []
        
        for t in tokens:
            if t in STREET_SUFFIX_MAP:
                normalized_tokens.append(STREET_SUFFIX_MAP[t])
            elif t in DIRECTION_MAP:
                normalized_tokens.append(DIRECTION_MAP[t])
            else:
                normalized_tokens.append(t)
                
        reconstructed = " ".join(normalized_tokens)
        # Clean up repeated commas and spacing
        reconstructed = re.sub(r"\s*,\s*", ", ", reconstructed)
        reconstructed = re.sub(r",+", ",", reconstructed).strip(" ,")
        return cls.RE_SPACES.sub(" ", reconstructed).strip()

class CountryNormalizer:
    """Open-set country normalization mapping known variants and standardizing unseen countries."""
    
    @classmethod
    def normalize(cls, country: Optional[str]) -> str:
        if not country:
            return ""
        c_clean = UnicodeUtils.clean(country).lower().strip()
        if c_clean in COUNTRY_MAP:
            return COUNTRY_MAP[c_clean]
        # Open-set fallback: Title Case if unrecognized
        return c_clean.title() if c_clean else


In [ ]:
# Quick sanity check on sample noisy cases
test_cases_name = [
    "Orelee's Barbershop",
    "B+ Retail Inc.",
    "International South Consultants Private Ltd",
    "Holloway Peak Inc Seafood",
    "WILFORDHANCOCK.COM"
]

print("=== NAME NORMALIZER SAMPLE OUTPUT ===")
for t in test_cases_name:
    print(f"Original:   '{t}'")
    print(f"Normalized: '{NameNormalizer.normalize(t)}'\n")

test_cases_addr = [
    "1795 Westchester Drive, High Point, NC",
    "KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",
    "2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",
    "Mack Rd, Haltom City, Texas"
]

print("=== ADDRESS NORMALIZER SAMPLE OUTPUT ===")
for t in test_cases_addr:
    print(f"Original:   '{t}'")
    print(f"Normalized: '{AddressNormalizer.normalize(t)}'\n")

test_cases_country = ["US", "usa", "India", "IND", "France", "fr", "Germany", ""]
print("=== COUNTRY NORMALIZER (OPEN-SET) SAMPLE OUTPUT ===")
for t in test_cases_country:
    print(f"Original: '{t}' --> Normalized: '{CountryNormalizer.normalize(t)}'")


## Step 4: Separate Normalization for S1, S2, and S3
Here we define **separate, dedicated normalization functions** for each source:
- `normalize_source1_record()`: Optimized for Source 1 records.
- `normalize_source2_record()`: Optimized for Source 2 records (handles high mojibake & uppercase noise).
- `normalize_source3_record()`: Optimized for Source 3 records (handles missing fields & URLs in names).

**Critical Constraint:** Raw fields are preserved side-by-side:
- `business_name_raw` & `business_name_normalized`
- `business_address_raw` & `business_address_normalized`
- `country_raw` & `country_normalized`


In [ ]:
def normalize_source1_record(row: Dict[str, str]) -> Dict[str, str]:
    """Dedicated normalizer for Source 1 records."""
    raw_name = row.get("business_name", "") or ""
    raw_addr = row.get("business_address", "") or ""
    raw_country = row.get("country", "") or ""
    
    return {
        "entity_id": row["entity_id"],
        "source": "source1",
        "business_name_raw": raw_name,
        "business_name_normalized": NameNormalizer.normalize(raw_name),
        "business_address_raw": raw_addr,
        "business_address_normalized": AddressNormalizer.normalize(raw_addr),
        "country_raw": raw_country,
        "country_normalized": CountryNormalizer.normalize(raw_country)
    }

def normalize_source2_record(row: Dict[str, str]) -> Dict[str, str]:
    """Dedicated normalizer for Source 2 records (extra care for mojibake & leading noise)."""
    raw_name = row.get("business_name", "") or ""
    raw_addr = row.get("business_address", "") or ""
    raw_country = row.get("country", "") or ""
    
    # Strip leading dashes / punctuation artifacts observed in S2
    clean_name = re.sub(r"^[-_\s]+", "", raw_name)
    
    return {
        "entity_id": row["entity_id"],
        "source": "source2",
        "business_name_raw": raw_name,
        "business_name_normalized": NameNormalizer.normalize(clean_name),
        "business_address_raw": raw_addr,
        "business_address_normalized": AddressNormalizer.normalize(raw_addr),
        "country_raw": raw_country,
        "country_normalized": CountryNormalizer.normalize(raw_country)
    }

def normalize_source3_record(row: Dict[str, str]) -> Dict[str, str]:
    """Dedicated normalizer for Source 3 records (handles domains / web names and missing addresses)."""
    raw_name = row.get("business_name", "") or ""
    raw_addr = row.get("business_address", "") or ""
    raw_country = row.get("country", "") or ""
    
    # If business name is a website (e.g. 'wilfordhancock.com'), strip domain extension for normalized form
    clean_name = raw_name
    if re.search(r"\.(com|org|net|in|co|io)$", raw_name, re.IGNORECASE):
        clean_name = re.sub(r"\.(com|org|net|in|co|io)$", "", raw_name, flags=re.IGNORECASE)
        
    return {
        "entity_id": row["entity_id"],
        "source": "source3",
        "business_name_raw": raw_name,
        "business_name_normalized": NameNormalizer.normalize(clean_name),
        "business_address_raw": raw_addr,
        "business_address_normalized": AddressNormalizer.normalize(raw_addr),
        "country_raw": raw_country,
        "country_normalized": CountryNormalizer.normalize(raw_country)
    }

print("Separate normalizers for S1, S2, and S3 successfully defined!")


In [ ]:
print("=== TESTING SEPARATE S1, S2, S3 NORMALIZATION ON GROUND TRUTH PAIRS ===\n")

for i, pair in enumerate(gt_pairs[:5], 1):
    s1_norm = normalize_source1_record({
        "entity_id": pair["s1_id"],
        "business_name": pair["s1_name"],
        "business_address": pair["s1_addr"],
        "country": pair["s1_country"]
    })
    
    match_row = {
        "entity_id": pair["match_id"],
        "business_name": pair["match_name"],
        "business_address": pair["match_addr"],
        "country": pair["match_country"]
    }
    match_norm = normalize_source2_record(match_row) if pair["match_source"] == "S2" else normalize_source3_record(match_row)
    
    print(f"PAIR #{i}:")
    print(f"  [RAW S1]        : '{s1_norm['business_name_raw']}' | '{s1_norm['business_address_raw']}'")
    print(f"  [RAW {pair['match_source']}]        : '{match_norm['business_name_raw']}' | '{match_norm['business_address_raw']}'")
    print(f"  [NORMALIZED S1] : '{s1_norm['business_name_normalized']}' | '{s1_norm['business_address_normalized']}'")
    print(f"  [NORMALIZED {pair['match_source']}]: '{match_norm['business_name_normalized']}' | '{match_norm['business_address_normalized']}'")
    print("-" * 100)


## Step 5: Full File Streaming Pipeline
Now we define the high-performance streaming processor that:
- Reads the raw TSV files in streaming chunks (handles multi-million rows without memory spikes).
- Applies `normalize_source1_record`, `normalize_source2_record`, or `normalize_source3_record` based on file type.
- Writes the normalized dataset to `data/processed/phase1/` in standard TSV format.
- Guarantees **Raw Data Preservation**.


In [ ]:
def process_source_file(input_file: Path, output_file: Path, normalizer_func, max_rows: Optional[int] = None):
    """Processes a full source TSV file row by row, ensuring high speed and low memory usage."""
    fieldnames = [
        "entity_id",
        "source",
        "business_name_raw",
        "business_name_normalized",
        "business_address_raw",
        "business_address_normalized",
        "country_raw",
        "country_normalized"
    ]
    
    print(f"\nProcessing: {input_file.name} --> {output_file.name}...")
    processed_count = 0
    
    with open(input_file, mode="r", encoding="utf-8", errors="replace") as fin, \
         open(output_file, mode="w", encoding="utf-8", newline="") as fout:
        
        reader = csv.DictReader(fin, delimiter="\t")
        writer = csv.DictWriter(fout, fieldnames=fieldnames, delimiter="\t")
        writer.writeheader()
        
        for row in reader:
            normalized_record = normalizer_func(row)
            writer.writerow(normalized_record)
            processed_count += 1
            
            if processed_count % 500000 == 0:
                print(f"  ...processed {processed_count:,} rows")
                
            if max_rows and processed_count >= max_rows:
                break
                
    print(f"Completed {input_file.name}: {processed_count:,} total records written to {output_file.name}.")
    return processed_count


## Step 6: Batch Execution for All 6 Files (Train & Test)
You can set `SAMPLE_RUN = True` to quickly test on 50,000 rows per file, or `SAMPLE_RUN = False` to run full normalization across all millions of rows.


In [ ]:
# Set to False to run all rows, or an integer (e.g. 50000) for a rapid sample run
MAX_ROWS_PER_FILE = 50000  # Change to None for full production run over all 12.5M rows

tasks = [
    # Train files
    (TRAIN_DIR / "train_source1.tsv", PROCESSED_DIR / "train_source1_normalized.tsv", normalize_source1_record),
    (TRAIN_DIR / "train_source2.tsv", PROCESSED_DIR / "train_source2_normalized.tsv", normalize_source2_record),
    (TRAIN_DIR / "train_source3.tsv", PROCESSED_DIR / "train_source3_normalized.tsv", normalize_source3_record),
    # Test files
    (TEST_DIR / "test_source1.tsv", PROCESSED_DIR / "test_source1_normalized.tsv", normalize_source1_record),
    (TEST_DIR / "test_source2.tsv", PROCESSED_DIR / "test_source2_normalized.tsv", normalize_source2_record),
    (TEST_DIR / "test_source3.tsv", PROCESSED_DIR / "test_source3_normalized.tsv", normalize_source3_record),
]

summary_stats = {}
for in_path, out_path, norm_func in tasks:
    if in_path.exists():
        count = process_source_file(in_path, out_path, norm_func, max_rows=MAX_ROWS_PER_FILE)
        summary_stats[in_path.name] = count
    else:
        print(f"Warning: {in_path} does not exist.")

print("\n=== PHASE 1 NORMALIZATION SUMMARY ===")
for filename, count in summary_stats.items():
    print(f"- {filename}: {count:,} records processed.")


## Step 7: Quality Verification & Raw Data Preservation Checks
We run assertions on the generated normalized files to prove:
1. Every record has its raw data 100% untouched (`raw_input == business_name_raw`).
2. No rows are missing or dropped.
3. Both raw and normalized values are populated and ready for Phase 2 feature extraction.


In [ ]:
def verify_raw_data_preservation(raw_file: Path, processed_file: Path, max_check: int = 10000):
    """Verifies that raw data fields are 100% preserved and never mutated."""
    print(f"Verifying raw data preservation between {raw_file.name} and {processed_file.name}...")
    
    with open(raw_file, mode="r", encoding="utf-8", errors="replace") as f_raw, \
         open(processed_file, mode="r", encoding="utf-8", errors="replace") as f_proc:
        
        r_reader = csv.DictReader(f_raw, delimiter="\t")
        p_reader = csv.DictReader(f_proc, delimiter="\t")
        
        checked = 0
        for r_row, p_row in zip(r_reader, p_reader):
            # Assert IDs match
            assert r_row["entity_id"] == p_row["entity_id"], f"ID mismatch: {r_row['entity_id']} vs {p_row['entity_id']}"
            # Assert raw fields match exactly
            assert (r_row.get("business_name") or "") == p_row["business_name_raw"], "Name mutation detected!"
            assert (r_row.get("business_address") or "") == p_row["business_address_raw"], "Address mutation detected!"
            assert (r_row.get("country") or "") == p_row["country_raw"], "Country mutation detected!"
            
            checked += 1
            if checked >= max_check:
                break
                
    print(f" PASSED: Verified {checked:,} records. Raw data is 100% preserved without mutation!")

# Run check on train_source1
verify_raw_data_preservation(TRAIN_DIR / "train_source1.tsv", PROCESSED_DIR / "train_source1_normalized.tsv")
